In [ ]:
import polars as pl
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from datetime import datetime

matplotlib.use("TkAgg")  # ou "QtAgg"
pl.Config.set_tbl_rows(-1)  # todas as linhas
pl.Config.set_tbl_cols(-1)
pl.Config.set_float_precision(2)
pl.Config.set_decimal_separator(',')
pl.Config.set_thousands_separator('.')

ENDERECO_DADOS = 'C:/DADOS/NOVO_BOLSA_FAMILIA/'
ENDERECO_VOTACAO = 'C:/DADOS/VOTACAO/'

In [ ]:
# obtendo os dados: Bolsa familia parquet e votação csv
try:
    print('Obtendo dados...')

    df_bolsa_familia = pl.scan_parquet(ENDERECO_DADOS + 'bolsa_familia.parquet')
    # df_bolsa_familia = df_bolsa_familia.collect()
    # print(df_bolsa_familia.columns)
    # print(df_bolsa_familia.head())
    
    df_dados_votacao = pl.read_csv(ENDERECO_VOTACAO + 'votacao_secao_2022_BR.csv', separator=';', encoding='iso-8859-1')
    print(df_dados_votacao.columns)
    # print(df_votacao.head())


    # Filtrar p/ o segundo turno 'NR_TURNO' e nº votável 13 e 22
    df_votacao_turno2 = df_dados_votacao.filter(
        (pl.col('NR_TURNO') == 2) &
        (pl.col('NR_VOTAVEL').is_in([13, 22]))
    )

    print('Dados obtidos com sucesso!')

except Exception as e:
    print(f'Erro ao ler os dados: {e}')

In [ ]:
# Iniciando o processamento Votação...
try:

    '''VOTAÇÃO'''
    # Delimitar as variáveis, converter p Categorical, agrupar/totalizar e ordenar
    print('Iniciando o processamento votação...')

    # Delimitando as variáveis em votação 'SG_UF', 'NM_VOTAVEL', 'QT_VOTOS'
    df_votacao = (
        df_votacao_turno2.lazy()
        .select(['SG_UF', 'NM_VOTAVEL', 'QT_VOTOS'])
    )
    print(df_votacao)

    # Converte p/ Categorical
    df_votacao = df_votacao.with_columns(
        pl.col('SG_UF').cast(pl.Categorical),
        pl.col('NM_VOTAVEL').cast(pl.Categorical)
    )

    # Agrupar / Totalizar os votos por candidato
    df_votacao = (
        df_votacao.group_by(['SG_UF', 'NM_VOTAVEL'])
        .agg(pl.col('QT_VOTOS').sum())
        .sort('SG_UF', descending=False)
    )

    # # # coleta os dados
    # df_votacao = df_votacao.collect()
    # display(df_votacao)

except Exception as e:
    print(f'Erro ao processar os dados de votação: {e}')

In [ ]:
# # Iniciando o processamento Bosla Familia...
try:
    '''BOLSA FAMILIA'''
    # Delimitar as variáveis, converter Categorical, agrupar/totalizar e ordenar

    print('Iniciando o processamento bolsa familia...')
    df_bolsa_familia = (
        df_bolsa_familia.lazy()
        .select(['UF', 'VALOR PARCELA'])
    )

    # Converte p/ Categorical
    df_bolsa_familia = df_bolsa_familia.with_columns(
        pl.col('UF').cast(pl.Categorical)
    )

    # Agrupar / Totalizar os valores
    df_bolsa_familia = (
        df_bolsa_familia.group_by('UF')
        .agg(pl.col('VALOR PARCELA').sum())
        .sort('UF', descending=False)
    )

    # coleta os dados
    # df_bolsa_familia = df_bolsa_familia.collect()
    # display(df_bolsa_familia)

except Exception as e:
    print(f'Erro ao processar os dados Bolsa Familia: {e}')

In [ ]:
# Merge Bolsa Familia e Votação
try:
    # Juntar os DataFrames c/ Join no Polars. É o Merge do Pandas
    df_votos_bolsa_familia = df_votacao.join(df_bolsa_familia, left_on='SG_UF', right_on='UF')
    display(df_votos_bolsa_familia.collect().sort('VALOR PARCELA', descending=True))
    df_votos_bolsa_familia = df_votos_bolsa_familia.collect()
    display(df_votos_bolsa_familia)

except Exception as e:
    print(f'Erro ao juntar os dataframes: {e}')

In [ ]:
# Calculando a correlação
try:
    print('Correlacionando os dados...')
    dict_correlacoes = {}

    # Faz o for, somente para candidatos únicos
    for candidato in df_votos_bolsa_familia['NM_VOTAVEL'].unique():
        # filtrar por candidato
        df_candidato = (
            df_votos_bolsa_familia.filter(pl.col('NM_VOTAVEL') == candidato)
        )
        # Criar array com os votos e os valores das parcelas
        array_votos = np.array(df_candidato['QT_VOTOS'])
        array_bolsa_familia = np.array(df_candidato['VALOR PARCELA'])

        # Calcular o coeficiente de correlação
        # O resultado é uma matriz
        correlacao = np.corrcoef(array_votos, array_bolsa_familia)[0, 1]
        print(f'Correlação do candidato {candidato}: {correlacao}')

        # Adicionar ao dicionário
        dict_correlacoes[candidato] = correlacao
        # print(dict_correlacoes)

except Exception as e:
    print(f'Erro ao calcular a correlação: {e}')

In [ ]:
# Visualizando os resultados
try:
    print('Plotando os resultados...')

    plt.subplots(2, 2, figsize=(17, 7))
    plt.suptitle('Votação x Bolsa Família', fontsize=16)



    # Posição 1: '''Ranking LULA'''
    '''# Posição 1 Ranking LULA'''
    plt.subplot(2, 2, 1)
    plt.title('Lula')

    df_lula = df_votos_bolsa_familia.filter(pl.col('NM_VOTAVEL') == 'LUIZ INÁCIO LULA DA SILVA')
    df_lula = df_lula.sort('QT_VOTOS', descending=True)

    # Gráfico de Colunas Lula
    plt.bar(df_lula['SG_UF'], df_lula['QT_VOTOS'])



    # Posição 2: Ranking Bolsonaro '''Ranking Bolsonaro'''
    '''# Posição 2 Ranking BOLSONARO'''
    plt.subplot(2, 2, 2)
    plt.title('Bolsonaro')

    df_bolsonaro = df_votos_bolsa_familia.filter(pl.col('NM_VOTAVEL') == 'JAIR MESSIAS BOLSONARO')
    df_bolsonaro = df_bolsonaro.sort('QT_VOTOS', descending=True)

    # Gráfico de Colunas Bolsonaro
    plt.bar(df_bolsonaro['SG_UF'], df_bolsonaro['QT_VOTOS'])



    # Posição 3: Ranking do bolsa família por UF
    '''# Posição 3 Ranking BOLSA FAMILIA'''
    plt.subplot(2, 2, 3)
    plt.title('Valor Parcela')

    # Gráfico de Colunas
    # df_bolsa_familia = df_bolsa_familia.collect()
    df_votos_bolsa_familia = df_votos_bolsa_familia.sort('VALOR PARCELA', descending=True)
    plt.bar(df_votos_bolsa_familia['SG_UF'], df_votos_bolsa_familia['VALOR PARCELA'])



    # Posição 4: Correlação
    '''# Posição 4 MEDIDAS DA CORRELAÇÃO'''
    plt.subplot(2, 2, 4)
    plt.title('Correlações')

    # coordenadas do plt.text
    x = 0.2
    y = 0.6

    for candidato, correlacao in dict_correlacoes.items():
        plt.text(x, y, f'{candidato}: {correlacao}', fontsize=12)

        # reduzir 0.2 do eixo Y
        # y = y - 0.2
        y -= 0.2
    
    plt.axis('off')
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f'Erro ao visualizar os resultados: {e}')


In [ ]:
# Gráfico de Dispersão Lula e Bolsonaro
try:
    print('Gerando gráficos de dispersão...')
    hora_inicio = datetime.now()

    plt.figure(figsize=(14, 6))
    plt.suptitle('Dispersão: Votos x Valor Parcela por Candidato')


    # Subplot para Lula
    plt.subplot(1, 2, 1)
    df_lula = df_votos_bolsa_familia.filter(pl.col('NM_VOTAVEL') == 'LUIZ INÁCIO LULA DA SILVA')
    plt.scatter(df_lula['QT_VOTOS'], df_lula['VALOR PARCELA'], alpha=0.7, color='blue')
    plt.title('Lula')
    plt.xlabel('QT_VOTOS')
    plt.ylabel('VALOR PARCELA')



    # Subplot para Bolsonaro
    plt.subplot(1, 2, 2)
    df_bolsonaro = df_votos_bolsa_familia.filter(pl.col('NM_VOTAVEL') == 'JAIR MESSIAS BOLSONARO')
    plt.scatter(df_bolsonaro['QT_VOTOS'], df_bolsonaro['VALOR PARCELA'], alpha=0.7, color='green')
    plt.title('Bolsonaro')
    plt.xlabel('QT_VOTOS')
    plt.ylabel('VALOR PARCELA')

    plt.tight_layout()
    plt.show()

    hora_fim = datetime.now()
    print('Gráficos de dispersão gerados com sucesso! Tempo de processamento: ', hora_fim - hora_inicio)

except Exception as e:
    print(f'Erro ao gerar gráficos de dispersão: {e}')